# 02a — Grid search de configuración del sample

Explora **72 combinaciones** de parámetros de filtrado del 02a_users.ipynb sobre
el dataset users.csv (cargado UNA SOLA VEZ) para encontrar la configuración óptima.

**No genera parquets**. Solo CSVs comparativos + heatmaps + recomendación.

## Grid (4 × 6 × 3 = 72)
- `CUTOFF_OFFSET_DAYS`: 45, 60, 90, 120
- `SPIKE_MAX_DAYS`: 7, 14, 30, 60, 90, 9999 (sin filtro)
- `MIN_NUM_LOGINS_BASIC`: 2, 5, 10

## 4 scores complementarios
- **S1**: balance puro = `min(rate, 1-rate) × 2` (1.0 = 50/50 perfecto)
- **S2**: balance × log10(N) — equilibrio entre balance y tamaño
- **S3**: min class size = `min(n_churn, n_no_churn)`
- **S4**: viability bool = `N≥20k ∧ 0.25≤rate≤0.65 ∧ min_class≥5k`

Ranking principal: **S2 sobre churn_30d**.


In [1]:
# [SETUP] Imports y paths
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from itertools import product
import time
import re

ROOT = Path.cwd().parent.parent
DATA_RAW = ROOT / 'data' / 'data_raw'
INFORMES = ROOT / 'informes' / 'fase1_cleaning' / 'grid_search_sample'
INFORMES.mkdir(parents=True, exist_ok=True)

print(f'ROOT      : {ROOT}')
print(f'DATA_RAW  : {DATA_RAW}')
print(f'INFORMES  : {INFORMES}')


ROOT      : /Users/jezquerro/Documents/tfg
DATA_RAW  : /Users/jezquerro/Documents/tfg/data/data_raw
INFORMES  : /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/grid_search_sample


In [2]:
# [EXEC] Cargar users.csv UNA SOLA VEZ (lógica replicada de 02a cell 4)
t0 = time.time()
csv_path = DATA_RAW / 'users.csv'
print(f'Cargando {csv_path} ...')

users_raw = pd.read_csv(csv_path, low_memory=False)
t_load = time.time() - t0
print(f'  Cargado en {t_load:.1f}s — shape: {users_raw.shape}')

# Extraer user_id desde ObjectId('...')  [mismo regex que 02a]
_id_col = '_id' if '_id' in users_raw.columns else users_raw.columns[0]
users_raw['user_id'] = (
    users_raw[_id_col].astype(str)
    .str.extract(r"ObjectId\(?'?([a-f0-9]+)'?\)?", expand=False)
    .fillna(users_raw[_id_col].astype(str))
)

# Parsear created_at_dt (tz-aware → naive, igual que 02a)
if 'created_at' in users_raw.columns:
    users_raw['created_at_dt'] = pd.to_datetime(
        users_raw['created_at'], errors='coerce', utc=True
    ).dt.tz_localize(None)

# Parsear last_login_dt (unix seconds → datetime naive, igual que 02a)
if 'last_login_date' in users_raw.columns:
    users_raw['last_login_dt'] = pd.to_datetime(
        users_raw['last_login_date'], unit='s', errors='coerce'
    )

# REFERENCE_DATE: max de last_login_dt (naive)
REFERENCE_DATE = users_raw['last_login_dt'].max()

print(f'\nREFERENCE_DATE: {REFERENCE_DATE}')
print(f'created_at_dt dtype: {users_raw["created_at_dt"].dtype}')
print(f'last_login_dt dtype: {users_raw["last_login_dt"].dtype}')
print(f'Filas totales: {len(users_raw):,}')
print(f'Memoria: {users_raw.memory_usage(deep=True).sum()/1e6:.1f} MB')


Cargando /Users/jezquerro/Documents/tfg/data/data_raw/users.csv ...


  Cargado en 3.7s — shape: (1164568, 30)



REFERENCE_DATE: 2026-04-04 18:23:37
created_at_dt dtype: datetime64[us]
last_login_dt dtype: datetime64[s]
Filas totales: 1,164,568


Memoria: 834.8 MB


In [3]:
# [EXEC] Función reutilizable: aplicar filtros + calcular targets/scores
def evaluate_config(users_raw, cutoff_offset_days, spike_max_days, min_num_logins_basic,
                    fixed_min_year=2018, fixed_min_lifespan=0, fixed_min_logins_eng=2):
    """Replica filtros del 02a y devuelve métricas del sample resultante."""
    cutoff_date = REFERENCE_DATE - pd.Timedelta(days=cutoff_offset_days)

    df = users_raw  # NO copiar — solo filtrar (reduce memoria, df_filtered es view-like)
    n_inicial = len(df)

    # F1: tutorial_done
    if 'tutorial_done' in df.columns:
        df = df[df['tutorial_done'] == True]
    # F2: last_login válida y > 2018
    df = df[df['last_login_dt'].notna() & (df['last_login_dt'].dt.year > 2018)]
    # F3: num_logins
    df = df[df['num_logins'] >= min_num_logins_basic]
    # F4: created_at year >= fixed_min_year
    df = df[df['created_at_dt'].dt.year >= fixed_min_year]
    # F5: created_at <= cutoff
    df = df[df['created_at_dt'] <= cutoff_date]

    n_after_quality = len(df)

    # Calcular days_since_last_login al cutoff (clip negativos a 0)
    df = df.copy()
    df['days_since_last_login'] = (
        ((cutoff_date - df['last_login_dt']).dt.total_seconds() / 86400)
        .round(0).astype(int).clip(lower=0)
    )

    # Filtro spike: days_since_last_login < spike_max_days
    df = df[df['days_since_last_login'] < spike_max_days]
    n_after_spike = len(df)

    # Filtros de engagement (lifespan + min_num_logins_engagement)
    last_login_clipped = df['last_login_dt'].clip(upper=cutoff_date)
    df['player_lifespan_days'] = (
        ((last_login_clipped - df['created_at_dt']).dt.total_seconds() / 86400)
        .round(0).astype(int)
    )
    df['has_corrupted_dates'] = df['player_lifespan_days'] < 0
    lifespan_ok = (df['player_lifespan_days'] >= fixed_min_lifespan) | df['has_corrupted_dates']
    logins_ok = df['num_logins'] >= fixed_min_logins_eng
    df = df[lifespan_ok & logins_ok]
    n_sample = len(df)

    # Targets
    targets = {}
    for window in [14, 30]:
        threshold = cutoff_date + pd.Timedelta(days=window)
        churn_col = (df['last_login_dt'] <= threshold)
        targets[f'churn_{window}d_rate'] = float(churn_col.mean()) if len(df) > 0 else 0.0
        targets[f'n_churners_{window}d'] = int(churn_col.sum())
        targets[f'n_non_churners_{window}d'] = int((~churn_col).sum())

    # Scores
    rate_30 = targets['churn_30d_rate']
    rate_14 = targets['churn_14d_rate']
    balance_30 = min(rate_30, 1 - rate_30) * 2 if 0 < rate_30 < 1 else 0.0
    balance_14 = min(rate_14, 1 - rate_14) * 2 if 0 < rate_14 < 1 else 0.0
    log_n = float(np.log10(max(n_sample, 1)))
    score_30 = balance_30 * log_n
    score_14 = balance_14 * log_n
    min_class_30 = min(targets['n_churners_30d'], targets['n_non_churners_30d'])
    min_class_14 = min(targets['n_churners_14d'], targets['n_non_churners_14d'])
    viable_30 = (n_sample >= 20_000) and (0.25 <= rate_30 <= 0.65) and (min_class_30 >= 5_000)
    viable_14 = (n_sample >= 20_000) and (0.25 <= rate_14 <= 0.65) and (min_class_14 >= 5_000)

    return {
        'cutoff_offset_days': cutoff_offset_days,
        'spike_max_days': spike_max_days,
        'min_num_logins_basic': min_num_logins_basic,
        'cutoff_date': cutoff_date.date().isoformat() if pd.notna(cutoff_date) else None,
        'n_inicial': n_inicial,
        'n_after_quality': n_after_quality,
        'n_after_spike': n_after_spike,
        'n_sample': n_sample,
        'pct_retained': round(n_sample / n_inicial * 100, 2),
        'churn_30d_rate': round(rate_30, 4),
        'churn_14d_rate': round(rate_14, 4),
        'n_churners_30d': targets['n_churners_30d'],
        'n_non_churners_30d': targets['n_non_churners_30d'],
        'n_churners_14d': targets['n_churners_14d'],
        'n_non_churners_14d': targets['n_non_churners_14d'],
        'min_class_size_30d': min_class_30,
        'min_class_size_14d': min_class_14,
        'balance_score_30d': round(balance_30, 4),
        'balance_score_14d': round(balance_14, 4),
        'score_balance_logn_30d': round(score_30, 4),
        'score_balance_logn_14d': round(score_14, 4),
        'viable_30d': viable_30,
        'viable_14d': viable_14,
    }

print('Función evaluate_config definida')


Función evaluate_config definida


In [4]:
# [EXEC] Grid search: 72 combinaciones
combinations = list(product(
    [45, 60, 90, 120],            # CUTOFF_OFFSET_DAYS
    [7, 14, 30, 60, 90, 9999],    # SPIKE_MAX_DAYS
    [2, 5, 10],                    # MIN_NUM_LOGINS_BASIC
))
print(f'Combinaciones a evaluar: {len(combinations)}')

results = []
t0 = time.time()
for i, (cutoff, spike, logins) in enumerate(combinations, 1):
    res = evaluate_config(users_raw, cutoff, spike, logins)
    results.append(res)
    if i % 12 == 0:
        elapsed = time.time() - t0
        print(f'  [{i:3d}/{len(combinations)}] cutoff={cutoff:>3} spike={spike:>5} logins={logins:>2} '
              f'→ N={res["n_sample"]:>7,} churn30={res["churn_30d_rate"]:.3f} | t={elapsed:.0f}s')

total_t = time.time() - t0
print(f'\nGrid completo en {total_t:.1f}s ({total_t/len(combinations):.2f}s por combo)')

results_df = pd.DataFrame(results)
results_df.to_csv(INFORMES / 'grid_search_results.csv', index=False)
print(f'Guardado: grid_search_results.csv ({len(results_df)} filas)')


Combinaciones a evaluar: 72


  [ 12/72] cutoff= 45 spike=   60 logins=10 → N= 28,051 churn30=0.886 | t=4s


  [ 24/72] cutoff= 60 spike=   14 logins=10 → N= 14,896 churn30=0.661 | t=9s


  [ 36/72] cutoff= 60 spike= 9999 logins=10 → N=283,053 churn30=0.982 | t=13s


  [ 48/72] cutoff= 90 spike=   60 logins=10 → N= 30,714 churn30=0.740 | t=17s


  [ 60/72] cutoff=120 spike=   14 logins=10 → N= 18,858 churn30=0.467 | t=21s


  [ 72/72] cutoff=120 spike= 9999 logins=10 → N=268,627 churn30=0.963 | t=26s

Grid completo en 25.7s (0.36s por combo)
Guardado: grid_search_results.csv (72 filas)


In [5]:
# [ANALYSIS] Configuraciones viables
viable_30 = results_df[results_df['viable_30d']].sort_values('score_balance_logn_30d', ascending=False)
viable_14 = results_df[results_df['viable_14d']].sort_values('score_balance_logn_14d', ascending=False)

print(f'Configuraciones viables (churn_30d): {len(viable_30)}/{len(results_df)}')
print(f'Configuraciones viables (churn_14d): {len(viable_14)}/{len(results_df)}')

if len(viable_30) > 0:
    viable_30.to_csv(INFORMES / 'viable_configs.csv', index=False)
    print('\n=== TOP 10 viables (por score_balance_logn_30d) ===')
    print(viable_30.head(10)[[
        'cutoff_offset_days', 'spike_max_days', 'min_num_logins_basic',
        'n_sample', 'churn_30d_rate', 'churn_14d_rate',
        'balance_score_30d', 'score_balance_logn_30d',
    ]].to_string(index=False))
else:
    # No viables: mostrar las menos malas por balance puro
    print('\nNO HAY CONFIGURACIONES VIABLES bajo umbrales (N≥20k, 0.25≤rate≤0.65, min_class≥5k)')
    print('\n=== TOP 10 menos malas (por balance_score_30d) ===')
    candidates = results_df.sort_values('balance_score_30d', ascending=False).head(10)
    print(candidates[[
        'cutoff_offset_days', 'spike_max_days', 'min_num_logins_basic',
        'n_sample', 'churn_30d_rate', 'churn_14d_rate',
        'balance_score_30d', 'score_balance_logn_30d',
    ]].to_string(index=False))
    # Guardar igual con flag
    pd.DataFrame(columns=results_df.columns).to_csv(INFORMES / 'viable_configs.csv', index=False)


Configuraciones viables (churn_30d): 8/72
Configuraciones viables (churn_14d): 17/72

=== TOP 10 viables (por score_balance_logn_30d) ===
 cutoff_offset_days  spike_max_days  min_num_logins_basic  n_sample  churn_30d_rate  churn_14d_rate  balance_score_30d  score_balance_logn_30d
                120               7                     5     25380          0.4709          0.3372             0.9418                  4.1480
                120              14                     5     29677          0.5475          0.4331             0.9050                  4.0476
                120               7                     2     34937          0.5593          0.4366             0.8814                  4.0043
                 90               7                     5     23603          0.5440          0.4020             0.9119                  3.9878
                120              30                    10     23061          0.5639          0.4672             0.8721                  3.8050
    

In [6]:
# [ANALYSIS] Top 10 por cada score
SCORE_COLS = ['n_sample', 'balance_score_30d', 'score_balance_logn_30d',
              'min_class_size_30d', 'balance_score_14d', 'score_balance_logn_14d']

# Top10 por el score principal: guardar como CSV
top10_main = results_df.nlargest(10, 'score_balance_logn_30d')[[
    'cutoff_offset_days', 'spike_max_days', 'min_num_logins_basic',
    'n_sample', 'churn_30d_rate', 'churn_14d_rate',
    'balance_score_30d', 'score_balance_logn_30d',
]]
top10_main.to_csv(INFORMES / 'top10_by_score.csv', index=False)

# Imprimir cada ranking (top 5 cada uno para no saturar)
for sc in SCORE_COLS:
    print(f'\n=== TOP 5 by {sc} ===')
    top5 = results_df.nlargest(5, sc)[[
        'cutoff_offset_days', 'spike_max_days', 'min_num_logins_basic',
        'n_sample', 'churn_30d_rate', 'balance_score_30d', sc,
    ]]
    print(top5.to_string(index=False))



=== TOP 5 by n_sample ===
 cutoff_offset_days  spike_max_days  min_num_logins_basic  n_sample  churn_30d_rate  balance_score_30d  n_sample
                 45            9999                     2    725238          0.9934             0.0131    725238
                 60            9999                     2    706811          0.9890             0.0220    706811
                 90            9999                     2    669504          0.9814             0.0372    669504
                120            9999                     2    627693          0.9755             0.0491    627693
                 45            9999                     5    502155          0.9918             0.0165    502155

=== TOP 5 by balance_score_30d ===
 cutoff_offset_days  spike_max_days  min_num_logins_basic  n_sample  churn_30d_rate  balance_score_30d  balance_score_30d
                 90               7                    10     15306          0.4773             0.9545             0.9545
               

In [7]:
# [ANALYSIS] Heatmaps cutoff × spike (mediana sobre min_num_logins_basic)
HEATMAPS = [
    ('churn_30d_rate', 'heatmap_churn_rate_30d.png', '.2%', 'viridis'),
    ('n_sample',       'heatmap_n_sample.png',       ',.0f', 'viridis'),
    ('score_balance_logn_30d', 'heatmap_balance_score.png', '.3f', 'RdYlGn'),
]

for metric, fname, fmt, cmap in HEATMAPS:
    pivot = results_df.pivot_table(
        values=metric, index='cutoff_offset_days', columns='spike_max_days',
        aggfunc='median',
    )
    fig, ax = plt.subplots(figsize=(10, 6))
    im = ax.imshow(pivot.values, cmap=cmap, aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel('SPIKE_MAX_DAYS')
    ax.set_ylabel('CUTOFF_OFFSET_DAYS')
    ax.set_title(f'{metric} (mediana sobre min_num_logins_basic)')

    # Anotar valores
    vals = pivot.values
    vmid = np.nanmedian(vals)
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            v = vals[i, j]
            if pd.notna(v):
                txt = format(v, fmt)
                color = 'white' if v < vmid else 'black'
                ax.text(j, i, txt, ha='center', va='center', color=color, fontsize=9)

    plt.colorbar(im, ax=ax, label=metric)
    plt.tight_layout()
    plt.savefig(INFORMES / fname, dpi=120, bbox_inches='tight')
    plt.close()
    print(f'Heatmap guardado: {fname}')


Heatmap guardado: heatmap_churn_rate_30d.png


Heatmap guardado: heatmap_n_sample.png
Heatmap guardado: heatmap_balance_score.png


In [8]:
# [REPORT] Generar recommendation.md
if len(viable_30) > 0:
    best = viable_30.iloc[0]
    status = 'VIABLE'
else:
    best = results_df.sort_values('balance_score_30d', ascending=False).iloc[0]
    status = 'MEJOR DENTRO DE LO INVIABLE (no cumple umbrales)'

current_match = results_df[
    (results_df['cutoff_offset_days'] == 60)
    & (results_df['spike_max_days'] == 90)
    & (results_df['min_num_logins_basic'] == 2)
]
current_str = (current_match[['n_sample','churn_30d_rate','churn_14d_rate',
                              'balance_score_30d','score_balance_logn_30d']]
               .to_string(index=False)
               if len(current_match) > 0
               else 'No presente en el grid (combo fuera del barrido)')

if len(viable_30) > 0:
    top5_str = viable_30.head(5)[[
        'cutoff_offset_days','spike_max_days','min_num_logins_basic',
        'n_sample','churn_30d_rate','score_balance_logn_30d',
    ]].to_string(index=False)
else:
    top5_str = 'Sin configuraciones viables — ver `grid_search_results.csv` para alternativas.'

md_content = f"""# Grid Search — Recomendación de configuración para 02a_users

**Fecha**: {time.strftime('%Y-%m-%d %H:%M')}
**Combinaciones evaluadas**: {len(results_df)}
**REFERENCE_DATE**: {REFERENCE_DATE}
**Configuraciones viables (churn_30d)**: {len(viable_30)}
**Configuraciones viables (churn_14d)**: {len(viable_14)}

## Configuración recomendada — {status}

```python
CUTOFF_OFFSET_DAYS    = {best['cutoff_offset_days']}
SPIKE_MAX_DAYS        = {best['spike_max_days']}
MIN_NUM_LOGINS_BASIC  = {best['min_num_logins_basic']}
```

### Métricas resultantes

| Métrica | Valor |
|---|---|
| N_SAMPLE | {best['n_sample']:,} |
| Pct retenido | {best['pct_retained']}% |
| Tasa churn_30d | {best['churn_30d_rate']:.2%} |
| Tasa churn_14d | {best['churn_14d_rate']:.2%} |
| N churners 30d | {best['n_churners_30d']:,} |
| N no-churners 30d | {best['n_non_churners_30d']:,} |
| Min class size 30d | {best['min_class_size_30d']:,} |
| Balance score 30d | {best['balance_score_30d']:.4f} (1.0 = perfecto 50/50) |
| Score balance × logN | {best['score_balance_logn_30d']:.4f} |

## Top 5 alternativas viables

{top5_str}

## Configuración actual del 02a (cutoff=60, spike=90, logins=2)

{current_str}

## Archivos generados

| Archivo | Contenido |
|---|---|
| `grid_search_results.csv` | Tabla completa con las 72 combinaciones |
| `top10_by_score.csv` | Top 10 ordenado por score principal |
| `viable_configs.csv` | Subset que pasa filtro de viabilidad |
| `heatmap_churn_rate_30d.png` | cutoff × spike, color = tasa churn 30d |
| `heatmap_n_sample.png` | cutoff × spike, color = N_sample |
| `heatmap_balance_score.png` | cutoff × spike, color = score compuesto |

## Próximo paso sugerido

1. Aplicar la configuración recomendada al notebook `02a_users.ipynb`:
   - Cambiar `CUTOFF_OFFSET_DAYS` en cell SETUP
   - Cambiar `SPIKE_MAX_DAYS` (cell 22 lo lee como filtro spike)
   - Cambiar `MIN_NUM_LOGINS_BASIC` (cell 10 lo lee como F3)
2. Re-lanzar la pipeline cutoff (orquestador en `scripts/run_pipeline_cutoff.py`)
"""

(INFORMES / 'recommendation.md').write_text(md_content)
print(md_content)


# Grid Search — Recomendación de configuración para 02a_users

**Fecha**: 2026-05-07 16:25
**Combinaciones evaluadas**: 72
**REFERENCE_DATE**: 2026-04-04 18:23:37
**Configuraciones viables (churn_30d)**: 8
**Configuraciones viables (churn_14d)**: 17

## Configuración recomendada — VIABLE

```python
CUTOFF_OFFSET_DAYS    = 120
SPIKE_MAX_DAYS        = 7
MIN_NUM_LOGINS_BASIC  = 5
```

### Métricas resultantes

| Métrica | Valor |
|---|---|
| N_SAMPLE | 25,380 |
| Pct retenido | 2.18% |
| Tasa churn_30d | 47.09% |
| Tasa churn_14d | 33.72% |
| N churners 30d | 11,951 |
| N no-churners 30d | 13,429 |
| Min class size 30d | 11,951 |
| Balance score 30d | 0.9418 (1.0 = perfecto 50/50) |
| Score balance × logN | 4.1480 |

## Top 5 alternativas viables

 cutoff_offset_days  spike_max_days  min_num_logins_basic  n_sample  churn_30d_rate  score_balance_logn_30d
                120               7                     5     25380          0.4709                  4.1480
                120          

In [9]:
# [REPORT] Logging dual: HTML
import sys
sys.path.insert(0, str(ROOT / 'scripts'))
try:
    from notebook_logging_template import export_notebook_to_html
    nb_path = ROOT / 'notebooks' / 'fase1_cleaning' / '02a_grid_search_sample_config.ipynb'
    html_path = INFORMES / 'grid_search_run.html'
    export_notebook_to_html(nb_path, html_path)
except Exception as e:
    print(f'Logging dual omitido (helpers no disponibles): {e}')

# Resumen final
print()
print('=' * 70)
print('GRID SEARCH COMPLETO')
print('=' * 70)
print(f'  Combinaciones evaluadas : {len(results_df)}')
print(f'  Viables (churn_30d)     : {len(viable_30)}')
print(f'  Viables (churn_14d)     : {len(viable_14)}')
print()
print(f'  Recomendación: cutoff={best["cutoff_offset_days"]}, '
      f'spike={best["spike_max_days"]}, logins={best["min_num_logins_basic"]}')
print(f'  → N={best["n_sample"]:,}, churn_30d={best["churn_30d_rate"]:.2%}, '
      f'balance_score={best["balance_score_30d"]:.3f}')
print()
print(f'  Outputs en: {INFORMES}')
print('=' * 70)


HTML generado: grid_search_run.html (0.4 MB)

GRID SEARCH COMPLETO
  Combinaciones evaluadas : 72
  Viables (churn_30d)     : 8
  Viables (churn_14d)     : 17

  Recomendación: cutoff=120, spike=7, logins=5
  → N=25,380, churn_30d=47.09%, balance_score=0.942

  Outputs en: /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/grid_search_sample
